# Phones, and the two ways across to words

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/s2t_pr_demo.ipynb) [![s2t_pr_demo](https://github.com/espnet/notebook/actions/workflows/s2t_pr_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/s2t_pr_demo.yml)

[POWSM](https://arxiv.org/abs/2510.24992) is a phonetic foundation model:
it hears speech and answers in phones. POWSM-CTC is the encoder-only
variant, and it does four things with one recording —

| task | you give | it answers |
| :-- | :-- | :-- |
| `<asr>` | the audio | the words |
| `<pr>` | the audio | the phones, in IPA |
| `<g2p>` | the audio **and the words** | the phones |
| `<p2g>` | the audio **and the phones** | the words |

— which is what this notebook runs, one after the other, on the same six
seconds of speech. CPU is enough.

## Install

In [ ]:
%pip install -q "espnet==202610.post2" espnet_model_zoo librosa

## A recording

In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O sample.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = librosa.load("sample.wav", sr=16000)
display(Audio(speech, rate=rate))

## The model

There are two POWSMs, and this notebook runs on either: change `MODEL` below
and nothing else. `powsm_ctc` reads its window in one pass and is what
`espnet phonemize` loads; `powsm` is the encoder-decoder, which searches and
is about four times slower on a CPU. Both write the same phone set — a
diphthong is two symbols in either, by design — so what differs between them
is which phones they choose, not what they can say.

`decode_window` is what makes the swap work: a CTC-only checkpoint is read
off its CTC head, one with a decoder is searched, and the caller does not
have to know which it is holding.

POWSM reads a 20-second window, so a shorter clip is padded to it. The
checkpoint also carries its own spelling of "work the language out yourself"
- `<unk>` here, where OWSM writes `<nolang>` - so ask the model rather than
typing a symbol.

In [ ]:
import librosa
from espnet2.bin.s2t_inference import Speech2Text

MODEL = "espnet/powsm_ctc"  # or "espnet/powsm", the encoder-decoder
s2t = Speech2Text.from_pretrained(MODEL, device="cpu")

window = s2t.preprocessor_conf["speech_length"]
nolang = s2t.no_language()
padded = librosa.util.fix_length(speech, size=rate * window)
print(f"{window} s window, language symbol {nolang}, CTC-only: {s2t.ctc_only}")

## `<pr>`: the phones

The task it is built for. POWSM writes each phone between slashes, so that
a phone spelled like a BPE token is still one token; spaced out is easier
to read and is what anything counting or aligning them wants.

In [ ]:
import re


def phones(decoded):
    """The phones out of POWSM's /p//h//o/ spelling."""
    return " ".join(re.findall(r"/([^/]+)/", decoded))


def run(task, text_prev="<na>"):
    """One window, decoded the way this checkpoint has to be."""
    decoded = s2t.decode_window(padded, nolang, task, text_prev)
    return decoded.split(">")[-1].strip() if "<" in decoded else decoded


said_in_phones = phones(run("<pr>"))
print(said_in_phones)

## `<asr>`: the words, for contrast

A phonetic model will transcribe. On English — where a model trained for
text has seen far more — it is the weaker choice, and what comes back below
shows it. On a language that general speech corpora barely cover, the
comparison is a different one: POWSM is competitive there, because it was
trained on a phone-annotated collection rather than on whatever happens to
be plentiful.

In [ ]:
print(run("<asr>"))

## `<g2p>`: phones for words you give

Now the audio is not the only input. The words you hand it ground the
answer — the phones come out for *those* words — and it is still listening
to the audio, which is where the pronunciation comes from.

In [ ]:
said = "the sale of the hotels is part of holiday's strategy"

print(phones(run("<g2p>", text_prev=said)))

## `<p2g>`: words for phones you give

The other direction. The prompt is phones, in the spelling the model was
trained on — each between slashes.

In [ ]:
as_prompt = "/" + "//".join(said_in_phones.split()) + "/"
print(as_prompt[:70], "...")

print(run("<p2g>", text_prev=as_prompt))

## Where next

- **From the terminal**: `espnet phonemize sample.wav` is `<pr>`, in one line
- **In the browser**: the [powsm-ctc Space](https://huggingface.co/spaces/espnet/powsm-ctc)
  offers all four tasks, and `espnet demo --model espnet/powsm_ctc` is the same page
- **From an assistant**: the MCP server's `phonemize` tool
- **Alignment**, with the same kind of model: [`s2t_align_demo.ipynb`](s2t_align_demo.ipynb)
- **The encoder-decoder POWSM**, [`espnet/powsm`](https://huggingface.co/espnet/powsm),
  answers the same four tasks with a search rather than a CTC pass